In [ ]:
import matplotlib.image as mpimg
import matplotlib.pyplot as plt
import numpy as np
import random

# some calculations are sloooow. set this to True to reduce samples width to 
# increase speed, when you only care about the later calculation
speedup = False

# noise control
apply_noise = True

# the paths of the images to be analyzed, comment
# some outto make the computation and debugging faster
paths = [
    #"01.png",
    #"02.png",
    #"03.png",
    #"04.png",
    #"05.png",
    #"06.png",
    "07.png",
]

ip = 0 # the path index for the single image graphs

# load images and show
print()
print(f"(A)")

S = [mpimg.imread(path) for path in paths]
H, W = S[ip].shape
N = H * W

for i in range(len(S)):
    s = S[i]
    
    if s.ndim == 3:
        s = s.mean(axis=2)
    
    if apply_noise:
        noise_max = (s.min() + s.max())
        for y in range(H):
            for x in range(W):
                noise = np.random.uniform(0, noise_max);
                s[y, x] = s[y, x] + noise
    
    S[i] = s
    
plt.imshow(S[ip], cmap="gray")
plt.title("$S(x, y)$")
plt.xlabel("x")
plt.ylabel("y")
plt.show()

print(f"min: {S[ip].min()} max: {S[ip].max()} mean: {S[ip].mean()}")

# zero mean
print()
print(f"(B)")

S = (S - np.mean(S)) / np.std(S)

print(f"min: {S[ip].min()} max: {S[ip].max()} mean: {S[ip].mean()}")

# correlation between neighboring pixels
print()
print(f"(C)")

steps = 1
if speedup:
    steps = 32

d_values = np.arange(0, W-1, steps)
y_values = np.zeros(len(d_values))
RS_values = []

for sindex, s in enumerate(S):
    RS_array = []
    
    for i, d in enumerate(d_values):
        print(f"progress... {sindex + 1}/{len(S)} {i + 1}/{len(d_values)}")
        
        temp = np.zeros((H, W - d))
        H_, W_ = temp.shape
        N_ = H_ * W_
        
        for y in range(H):
            for x in range(W - d):
                temp[y, x] = s[y, x] * s[y, x + d] / N_
        
        RS = temp.sum()
        y_values[i] += RS
        RS_array.append(RS)
        
    RS_values.append(RS_array)

print(f"RS_values: {RS_values}")

y_values /= len(S)

plt.plot(d_values, y_values)
plt.title("correlation of neighboring pixels")
plt.xlabel("distance $|x_1 - x_2|$ (pixels)")
plt.ylabel("$R^S(x_1-x_2)$")
plt.show()

## furiour power specturm 1-D
#print()
#print(f"(D)")
#
#steps = 1
#if speedup:
#    steps = 16
#
#L = np.arange(0, int(W/2), steps)
#
#x_values = []
#y_values = []
#
#for sindex, s in enumerate(S):
#
#    for i, l in enumerate(L):
#        k = l * 2 * np.pi / W
#        print(f"progress... {sindex + 1}/{len(S)} {i + 1}/{len(L)}")
#        
#        sum_c = 0
#        sum_s = 0
#        
#        for y in range(H):
#            for x in range(W):
#                val_c = s[y, x] * np.cos(k * x)
#                val_s = s[y, x] * np.sin(k * x)
#                
#                sum_c += val_c
#                sum_s += val_s
#        
#        s2 = sum_c * sum_c + sum_s * sum_s
#        
#        x_values.append(l)
#        y_values.append(s2)
#
#print("scatter...")
#
#plt.scatter(x_values, y_values, s=1)
#plt.title("Fourier power spectrum (1D)")
#plt.xlabel("$|k|$ (cycles/image)")
#plt.ylabel("signal power $|S(k)|^2$")
#plt.xscale('log')
#plt.yscale('log')
#plt.show()

# furiour power specturm 2-D
print()
print(f"(E)")

plt.imshow(S[ip], cmap="gray")
plt.title("original image $S$")
plt.xlabel("x")
plt.ylabel("y")
plt.show()

SFurier = np.fft.fft2(S[ip])
SFurierShifted = np.fft.fftshift(SFurier)

kx = np.fft.fftshift(np.fft.fftfreq(W)) * W
ky = np.fft.fftshift(np.fft.fftfreq(H)) * H
im = plt.imshow(
    np.log(np.abs(SFurierShifted)),
    extent=[kx[0], kx[-1], ky[0], ky[-1]],
)
plt.title("Fourier components $\log |\mathcal{S}(k)|$")
plt.xlabel("$k_x$ (cycles/image)")
plt.ylabel("$k_y$ (cycles/image)")
plt.colorbar(im)
plt.show()

S_ = np.real(np.fft.ifft2(SFurier))

plt.imshow(S_, cmap="gray")
plt.title("after the inverse Fourier transform")
plt.xlabel("x")
plt.ylabel("y")
plt.show()

print(f"sanity check: the difference should be 0 (due to rounding errors, close to 0")
Sdiff = S[ip] - S_
im = plt.imshow(Sdiff)
plt.title("$S - S'$")
plt.xlabel("x")
plt.ylabel("y")
plt.colorbar(im)
plt.show()

print(f"find magnitude related to distance...")
h, w = SFurierShifted.shape
center_x = w / 2
center_y = h / 2

x_values = []
y_values = []
values = []

for y in range(SFurierShifted.shape[0]):
    for x in range(SFurierShifted.shape[1]):
        dx = center_x - x
        dy = center_y - y
        distance = np.sqrt(dx * dx + dy * dy)

        x_value = distance
        y_value = np.abs(SFurierShifted[y, x])

        x_values.append(x_value)
        y_values.append(y_value)
        values.append((x_value, y_value))

print(f"sort...")

values = sorted(values, key=lambda t: t[0])

print(f"compute moving average...")
average_x_values = []
average_y_values = []

i = 0

while i < len(values):
    window_left = i
    window_right = i
    while window_right < len(values):
        min = values[window_left][0]
        max = values[window_right][0]
        if max - min < 1:
            window_right += 1
        else:
            break
    
    x_value_left = values[window_left][0]
    x_value_right = values[window_right - 1][0]
    average_x = (x_value_left + x_value_right) / 2

    summed = 0
    count = window_right - window_left

    for j in range(count):
        index = window_left + j
        value = values[index]
        summed += value[1]
    
    average_y = summed / count

    average_x_values.append(average_x)
    average_y_values.append(average_y)

    i = int((window_left + window_right) / 2) + 1

x_line = np.linspace(values[0][0], values[-1][0], 1000)
y_line = [1/(x**2) * np.exp(12) for x in x_line]

plt.scatter(x_values, y_values, s=1, color="#3836e0")
plt.plot(average_x_values, average_y_values, color="#2df414")
plt.plot(x_line, y_line, color="#ff0011", linestyle="--")
plt.xscale("log")
plt.yscale("log")
plt.title("$|S(k)|$ and $\sim 1/k^2$ (red) versus $|k|$")
plt.xlabel("$|k|$ (cycles/image)")
plt.ylabel("$|\mathcal{S}(k)|$")
plt.show()
    
# noise
print()
print(f"(F)")
print(f"noise control is at the top of the code")

# gain
print()
print(f"(G)")

for RSidx, RS in enumerate(RS_values):
    
    
    window = 5
    smooth_slope = np.convolve(np.abs(RS), np.ones(window)/window, mode="same")
    threshold = np.percentile(smooth_slope, 20)
    knee_idx = np.where(smooth_slope < threshold)[0][0]

    print(f"knee: {knee_idx}")

    for i, l in enumerate(L):
        #k = l * 2 * np.pi / W


        g = gain(l, 0, knee_idx)
        g_low = gain(l, 0, knee_idx / 2)
        g_high = gain(l, 0, knee_idx * 1.5)
    
    def gain(x, k_0, k_low):
        return (x + k_0) * np.exp(-(x*x)/(k_low * k_low))

    k_lows = [
        knee_idx / 4,
        knee_idx / 2,
        knee_idx,
        knee_idx * 2,
    ]

    for k_low in k_lows:
        print(f"$k_{{low}}={k_low}$")

        SFurier = np.fft.fft2(S[RSidx])
        gs = np.zeros(SFurier.shape)

        for y in range(SFurier.shape[0]):
            for x in range(SFurier.shape[1]):
                k_abs = np.sqrt(x*x+y*y)
                g_abs = np.sqrt((x - SFurier.shape[1] / 2)**2 + (y - SFurier.shape[0] / 2)**2)
                SFurier[y, x] *= gain(k_abs, 0, k_low)
                gs[y, x] = gain(g_abs, 0, k_low)

        S_ = np.real(np.fft.ifft2(SFurier))
        g_ = np.log(np.abs(np.fft.fftshift(np.fft.ifft2(gs))))

        fig, ax = plt.subplots(1, 2)

        ax[0].imshow(S[ip], cmap="gray")
        ax[0].set_title("$S$")
        ax[0].set_xlabel("$x$")
        ax[0].set_ylabel("$y$")

        ax[1].imshow(S_, cmap="gray")
        ax[1].set_title(f"$S' = UgK_O S$")
        ax[1].set_xlabel("$x$")
        ax[1].set_ylabel("$y$")

        plt.subplots_adjust(wspace=0.5, hspace=0.5)
        plt.show()

        fig, ax = plt.subplots(1, 2)

        im = ax[0].imshow(gs)
        ax[0].set_title(f"$g$")
        ax[0].set_xlabel("$k_x$ (cycles/image)")
        ax[0].set_ylabel("$k_y$ (cycles/image)")
        fig.colorbar(im, ax=ax[0])

        im = ax[1].imshow(g_)
        ax[1].set_title(f"$g' = \log |Ug|$")
        ax[1].set_xlabel("$x$")
        ax[1].set_ylabel("$y$")
        fig.colorbar(im, ax=ax[1])

        plt.subplots_adjust(wspace=0.5, hspace=0.5)
        plt.show()

#####################
print()
print("reached end!")